# 📚 StudyMate: Local AI Student Assistant (No External APIs)

This notebook runs entirely locally, with no external API required (no Gemini API):

1. **Layout-Aware OCR:** YOLO (layout detection) + Tesseract OCR — YOLO locates paragraphs/tables/figures on the page, then Tesseract reads each text region separately for higher accuracy
2. **Text Summarization:** BART (`facebook/bart-large-cnn`) with a Map-Reduce strategy (summarizes the full content, not just the first chunk)
3. **Local RAG & Brain:** ChromaDB + SentenceTransformers (`all-MiniLM-L6-v2`) + local LLM (`Qwen2.5-1.5B-Instruct`)
4. **Interactive UI:** PyVis (mind maps) + Gradio dashboard (bilingual — supports Arabic and English input)

---
### ⚠️ Notes from review and cleanup
- **TrOCR removed entirely**: it was referenced in the original version but wasn't actually being used. Since Tesseract already covers OCR, it is now the sole text-extraction engine.
- **YOLO is now a core part of the pipeline**: previously loaded but never used. It now detects text regions (layout detection) before OCR runs, which improves accuracy especially on pages with tables/figures next to text.
- **Summarization (BART) is now Map-Reduce**: it used to summarize only the first 3 chunks and silently drop the rest. It now summarizes the entire content.
- **Removed all dead code** (e.g. `ocr_handwriting`, which was never called from anywhere).

## 📦 Cell 1: Install Dependencies

In [ ]:
!pip install -q ultralytics pytesseract pymupdf pillow sentencepiece tiktoken
!pip install -q transformers sentence-transformers chromadb
!pip install -q pyvis gradio
!pip install -q spacy networkx
!python -m spacy download en_core_web_sm -q
!apt-get update -y && apt-get install -y tesseract-ocr tesseract-ocr-ara


## 📁 Cell 2: Mount Google Drive & Environment Setup

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Path to the folder on Google Drive
DRIVE_FOLDER = '/content/drive/MyDrive/student assistant'

if os.path.exists(DRIVE_FOLDER):
    print("✅ Drive mounted successfully! Files in folder:")
    print(os.listdir(DRIVE_FOLDER))
else:
    print(f"⚠️ Folder '{DRIVE_FOLDER}' not found. Creating it...")
    os.makedirs(DRIVE_FOLDER, exist_ok=True)


## 🔍 Cell 3: Load the YOLO Layout Detection Model

A `best.pt` file (your trained model) must be present in the Drive folder.
If it's missing, the code won't fail — it automatically falls back to full-page OCR without segmentation.

In [ ]:
import subprocess, sys, importlib

# Auto-check: install ultralytics automatically if missing (e.g. after a runtime restart post-install cell)
try:
    import ultralytics
except ImportError:
    print("⚠️ ultralytics not installed — installing automatically...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    importlib.invalidate_caches()

import torch
from ultralytics import YOLO

YOLO_PATH = os.path.join(DRIVE_FOLDER, 'best.pt')

if os.path.exists(YOLO_PATH):
    yolo_model = YOLO(YOLO_PATH)
    print("✅ YOLO layout model loaded successfully.")
else:
    yolo_model = None
    print("⚠️ 'best.pt' not found in Drive folder. Will fall back to full-page OCR (no layout detection).")


## 🔤 Cell 4: Set Up Tesseract OCR

Note: TrOCR was removed from here because it wasn't working, and Tesseract covers the same function (reading printed/handwritten text from an image).

In [ ]:
import pytesseract
from PIL import Image

# Check Tesseract version
print("Tesseract Version:", pytesseract.get_tesseract_version())

def ocr_printed(image: Image.Image, lang='eng+ara') -> str:
    """Extract text from an image using Tesseract (supports English + Arabic)"""
    return pytesseract.image_to_string(image, lang=lang, config='--psm 6')

print("✅ Tesseract OCR ready!")


## 🧩 Cell 5: Combine YOLO + Tesseract (Layout-Aware OCR)

This is the part that makes YOLO an actual part of the pipeline:

1. `detect_layout_regions`: runs YOLO on the page image and returns bounding boxes sorted top to bottom
2. `ocr_page_with_layout`: crops each text region and runs Tesseract on it separately, skipping figure/table regions so they don't corrupt the extracted text, and returns the best combined result. If YOLO is unavailable or finds nothing, it automatically falls back to full-page OCR.

⚠️ **Important:** the class names in `NON_TEXT_LABELS` must match the class names your `best.pt` was trained on (check `yolo_model.names` to confirm the correct names for your model).

In [ ]:
# Class names that are not text (figures/tables) — edit this list if your model's class names differ
NON_TEXT_LABELS = {"picture", "figure", "image", "table", "chart"}

def detect_layout_regions(image: Image.Image, conf_threshold=0.25):
    """Run YOLO on the page image, return bounding boxes sorted top to bottom"""
    if yolo_model is None:
        return []

    results = yolo_model.predict(source=image, conf=conf_threshold, verbose=False)
    regions = []
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0].item())
            label = r.names[cls_id]
            x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
            regions.append({"label": label, "bbox": (x1, y1, x2, y2)})

    # Sort regions from top to bottom of the page (approximate 50px row buckets)
    regions.sort(key=lambda r: (round(r["bbox"][1] / 50), r["bbox"][0]))
    return regions


def ocr_page_with_layout(image: Image.Image) -> str:
    """
    If YOLO is available: splits the page into regions and OCRs each text region separately.
    If unavailable or no regions found: falls back to full-page OCR.
    """
    regions = detect_layout_regions(image)
    if not regions:
        return ocr_printed(image)

    text_parts = []
    for region in regions:
        label = region["label"].lower()
        x1, y1, x2, y2 = region["bbox"]

        if label in NON_TEXT_LABELS:
            continue  # skip figures/tables when extracting text

        crop = image.crop((x1, y1, x2, y2))
        crop_text = ocr_printed(crop)
        if crop_text.strip():
            text_parts.append(crop_text.strip())

    combined = "\n".join(text_parts)
    return combined if combined.strip() else ocr_printed(image)

print("✅ Layout-aware OCR ready (YOLO + Tesseract)!")


## 📄 Cell 6: Load the Local LLM (Qwen2.5-1.5B-Instruct)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("Loading Local LLM (Qwen2.5-1.5B-Instruct)...")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

llm_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.9
)

def query_local_llm(prompt: str) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful and precise educational AI assistant."},
        {"role": "user", "content": prompt}
    ]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = llm_pipeline(formatted_prompt)
    response = outputs[0]['generated_text'][len(formatted_prompt):].strip()
    return response

print("✅ Local LLM loaded successfully!")


## 📖 Cell 7: PDF Content Extraction Pipeline (Smart PDF Extraction)

Difference from the original version: scanned PDF pages now use `ocr_page_with_layout` (YOLO + Tesseract) instead of plain full-page OCR.

In [ ]:
import fitz  # PyMuPDF
import io

def extract_pdf_content(pdf_path: str, start_page=1, end_page=None) -> str:
    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    if end_page is None or end_page > total_pages:
        end_page = total_pages

    full_text = []

    for page_num in range(start_page - 1, end_page):
        page = doc[page_num]
        text = page.get_text()

        # If the page is an image or has very little text, fall back to layout-aware OCR
        if len(text.strip()) < 50:
            pix = page.get_pixmap()
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            ocr_text = ocr_page_with_layout(img)
            text = f"[OCR Page {page_num + 1}]\n" + ocr_text

        full_text.append(f"--- Page {page_num + 1} ---\n{text}")

    return "\n\n".join(full_text)

print("✅ PDF extraction pipeline ready!")


## 📝 Cell 8: Summarization with BART (Map-Reduce)

Difference from the original version: instead of summarizing only the first 3 chunks and silently dropping the rest, it now summarizes **all** the content (map), and if the lecture is long (more than 3 chunks) it runs an additional summarization pass over the partial summaries (reduce) to keep the final output concise.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading BART Summarizer...")
MODEL_NAME = "facebook/bart-large-cnn"

bart_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
bart_model.to(device)

print(f"✅ BART Summarizer loaded on {device}!")

def _bart_summarize_chunk(chunk: str, max_length=130, min_length=30) -> str:
    inputs = bart_tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True).to(device)
    summary_ids = bart_model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )
    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def summarize_text(text: str, max_chunk_chars=2000) -> str:
    if not text.strip():
        return "No text available to summarize."

    # Map: summarize every chunk of text (not just the first 3 like before)
    chunks = [text[i:i + max_chunk_chars] for i in range(0, len(text), max_chunk_chars)]
    partial_summaries = [_bart_summarize_chunk(chunk) for chunk in chunks]

    combined = "\n".join(partial_summaries)

    # Reduce: if the lecture is long, run a final summarization pass over the partial summaries
    if len(partial_summaries) > 3:
        return _bart_summarize_chunk(combined, max_length=180, min_length=40)

    return combined


## 🧠 Cell 9: RAG & Vector DB (ChromaDB)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

print("Loading Embedding Model & Vector DB...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()

def setup_rag_index(text: str):
    if not text or not text.strip():
        return None

    # Clear any existing collection
    try:
        chroma_client.delete_collection("study_docs")
    except Exception:
        pass

    collection = chroma_client.create_collection("study_docs")

    # Chunking (100-character overlap between chunks so context isn't lost at the boundaries)
    chunks = [text[i:i + 500] for i in range(0, len(text), 400)]
    if not chunks:
        return None

    embeddings = embedder.encode(chunks).tolist()
    ids = [f"chunk_{i}" for i in range(len(chunks))]

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids
    )
    return collection


def rag_query(collection, query: str) -> str:
    if not collection:
        return "No document index found. Please upload a file first."

    query_emb = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=3)
    retrieved_docs = "\n".join(results['documents'][0])

    prompt = f"""Use the following contexts to answer the student's question accurately.

Contexts:
{retrieved_docs}

Question: {query}
Answer:"""
    return query_local_llm(prompt)

print("✅ RAG pipeline ready!")


## 🗺️ Cell 10: Quiz Generator & Mind Map

In [ ]:
import json
from pyvis.network import Network

def generate_quiz(text: str) -> str:
    prompt = f"""Generate a 3-question Multiple Choice Quiz (MCQ) based on this text.
Provide options A, B, C, D and explicitly state the correct answer.

Text:
{text[:1500]}
"""
    return query_local_llm(prompt)


def generate_mindmap_html(text: str, output_path="mindmap.html") -> str:
    prompt = f"""Extract main concepts from the text and output ONLY a valid JSON object.
Format:
{{
  "main_topic": "Topic Name",
  "branches": [
    {{"name": "Subtopic 1", "leafs": ["Detail A", "Detail B"]}},
    {{"name": "Subtopic 2", "leafs": ["Detail C"]}}
  ]
}}

Text:
{text[:1200]}
"""
    response = query_local_llm(prompt)

    net = Network(height="450px", width="100%", bgcolor="#0f172a", font_color="#f1f5f9", directed=True)

    try:
        json_start = response.find("{")
        json_end = response.rfind("}") + 1
        data = json.loads(response[json_start:json_end])

        root_title = data.get("main_topic", "Subject")
        net.add_node("Root", label=root_title, color="#6366f1", size=34, shape="dot")

        for i, branch in enumerate(data.get("branches", [])):
            b_id = f"b_{i}"
            net.add_node(b_id, label=branch["name"], color="#f59e0b", size=22, shape="dot")
            net.add_edge("Root", b_id, color="#64748b")

            for j, leaf in enumerate(branch.get("leafs", [])):
                l_id = f"l_{i}_{j}"
                net.add_node(l_id, label=leaf, color="#10b981", size=14, shape="dot")
                net.add_edge(b_id, l_id, color="#64748b")

    except Exception:
        # Failed to parse JSON from the LLM response — show a clear signal to the user instead of a silent fake fallback
        net.add_node("Root", label="⚠️ Could not automatically extract concepts", color="#ef4444", size=30)

    net.write_html(output_path)
    return output_path

print("✅ Quiz & Mind Map generators ready!")


## 🎓 Cell 10.5: Classical NLP Techniques (Non-LLM Alternative)

This section is completely independent of Qwen/BART — it relies entirely on classical NLP techniques, so there's real coverage of NLP methods here (not just "I used an off-the-shelf LLM"):

1. **TextRank Summarization**: extractive (not generative) summarization — uses the same `embedder` (SentenceTransformer) used in RAG, builds a sentence similarity graph (cosine similarity), and runs the **PageRank algorithm** over it to pick the most important sentences that actually exist in the text.
2. **Cloze Quiz Generation**: generates fill-in-the-blank questions via **POS tagging / NER** (spaCy) to extract key terms, then removes them from the original sentence and builds multiple-choice options from other terms in the same document — with no LLM text generation involved.

⚠️ Note: spaCy's `en_core_web_sm` is an **English** model. If your lectures are in Arabic, this technique performs weaker on Arabic text (English NER/POS won't recognize Arabic terms with the same accuracy). If most of your documents are Arabic, this could be swapped for an Arabic model (spaCy has no strong official Arabic model, so this would need a library like `camel-tools`, or you could rely solely on TextRank, which works across languages since it's embedding-based).

In [ ]:
import re
import numpy as np
import networkx as nx


def split_into_sentences(text: str):
    """Simple sentence splitter; supports both Arabic and English punctuation"""
    sentences = re.split(r'(?<=[.!?؟])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 20]


def textrank_summarize(text: str, num_sentences: int = 5) -> str:
    """
    Extractive summarization using TextRank:
    1) Embed each sentence
    2) Build a sentence similarity matrix (cosine similarity)
    3) Build a graph and run PageRank on it to rank sentence importance
    4) Return the top N sentences in their original order in the text
    """
    sentences = split_into_sentences(text)
    if len(sentences) <= num_sentences:
        return text

    embeddings = embedder.encode(sentences)

    sim_matrix = np.zeros((len(sentences), len(sentences)))
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                a, b = embeddings[i], embeddings[j]
                sim_matrix[i][j] = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

    graph = nx.from_numpy_array(sim_matrix)
    scores = nx.pagerank(graph)

    ranked = sorted(((scores[i], i, s) for i, s in enumerate(sentences)), reverse=True)
    top_sentences = sorted(ranked[:num_sentences], key=lambda x: x[1])

    return " ".join(s for _, _, s in top_sentences)


print("✅ TextRank summarizer ready!")


In [ ]:
import spacy
import random

nlp_spacy = spacy.load("en_core_web_sm")


def extract_keywords(text: str, top_n=15):
    """Extract key terms (NER + noun chunks) via POS tagging"""
    doc_spacy = nlp_spacy(text[:5000])  # cap for performance
    candidates = set()

    for ent in doc_spacy.ents:
        if len(ent.text.split()) <= 3:
            candidates.add(ent.text)

    for chunk in doc_spacy.noun_chunks:
        if 1 <= len(chunk.text.split()) <= 3:
            candidates.add(chunk.text.strip())

    return list(candidates)[:top_n]


def generate_cloze_quiz(text: str, num_questions: int = 3) -> str:
    """
    Classical fill-in-the-blank quiz generation:
    1) Extract key terms (NER/noun chunks) from the text
    2) Pick sentences containing those terms and remove the term (cloze deletion)
    3) Build answer options from other terms in the same document as distractors
    """
    sentences = split_into_sentences(text)
    keywords = extract_keywords(text)

    if not keywords:
        return "⚠️ Not enough key terms were found to build a quiz from this text."

    random.shuffle(sentences)
    questions = []
    used_keywords = set()

    for sentence in sentences:
        if len(questions) >= num_questions:
            break
        for kw in keywords:
            if kw in sentence and kw not in used_keywords and len(sentence) < 300:
                blanked = sentence.replace(kw, "______", 1)

                distractors = [k for k in keywords if k != kw]
                random.shuffle(distractors)
                options = distractors[:3] + [kw]
                if len(options) < 2:
                    continue
                random.shuffle(options)

                correct_letter = "ABCD"[options.index(kw)]
                q_text = f"Q{len(questions) + 1}: {blanked}\n"
                for letter, opt in zip("ABCD", options):
                    q_text += f"   {letter}) {opt}\n"
                q_text += f"   ✅ Correct answer: {correct_letter}) {kw}\n"

                questions.append(q_text)
                used_keywords.add(kw)
                break

    if not questions:
        return "⚠️ No suitable sentences were found to build a Cloze quiz from this text."

    return "\n".join(questions)


print("✅ Cloze quiz generator ready!")


## 🌐 Cell 11: Gradio Dashboard

Same functionality as before, plus:
- Actual file-type validation before building the RAG index (instead of building an index on an error message)
- Clearer status messages for the user (text extraction success/failure)
- Better design: gradient header, cards, icons, a font that also renders Arabic well, and a preview of the first part of the extracted text

In [ ]:
import gradio as gr

# State Management
APP_STATE = {
    "extracted_text": "",
    "collection": None
}

SUPPORTED_EXTENSIONS = ["pdf", "png", "jpg", "jpeg"]


def handle_file_upload(file_obj):
    if file_obj is None:
        return "⚠️ No file was uploaded.", ""

    file_path = file_obj.name
    ext = file_path.split(".")[-1].lower()

    if ext not in SUPPORTED_EXTENSIONS:
        return f"❌ Unsupported format: .{ext} — please upload a PDF or an image (PNG/JPG).", ""

    if ext == "pdf":
        text = extract_pdf_content(file_path)
    else:
        img = Image.open(file_path)
        text = ocr_page_with_layout(img)

    if not text or not text.strip():
        return "⚠️ No text could be extracted from the file. Try a clearer image.", ""

    APP_STATE["extracted_text"] = text
    APP_STATE["collection"] = setup_rag_index(text)

    preview = text[:400] + ("..." if len(text) > 400 else "")
    return f"✅ Successfully extracted {len(text)} characters!", preview


def handle_explain_request(user_message: str, full_text: str, max_chars: int = 8000) -> str:
    """
    Uses the local LLM (Qwen) to understand the student's actual request
    (specific pages / a specific section / a general explanation) instead of
    blindly summarizing with BART regardless of what the message says.
    The text contains ready-made page markers (--- Page N ---) from
    extract_pdf_content, so the LLM can determine the requested range itself.
    """
    # Max size of text sent to the LLM (context window) — if the document is
    # longer than this it gets truncated; full coverage of a very long
    # document would need to be processed in batches
    context_text = full_text[:max_chars]
    truncated_note = "" if len(full_text) <= max_chars else \
        "\n\n[Note: the document is very long; only part of it was shown to the model.]"

    prompt = f"""You are a helpful study assistant. The student's exact request is:
"{user_message}"

The document below is divided into pages marked as "--- Page N ---".
- If the student asked about specific pages or a specific section, answer using ONLY that part.
- If the student asked for a general explanation or summary, cover the whole document below.
- Respond in the same language the student used in their request.

Document:
{context_text}{truncated_note}
"""
    return query_local_llm(prompt)


def user_chat_router(user_message, history):
    text = APP_STATE["extracted_text"]
    coll = APP_STATE["collection"]

    if not text:
        return "Please upload a document or lecture image first!"

    msg_lower = user_message.lower()

    if any(word in msg_lower for word in ["شرح", "اشرح", "ملخص", "تلخيص", "explain", "summarize", "overview"]):
        # We send the student's actual message + the text to the LLM (not BART) so it
        # understands the request — whether specific pages, a specific part, or a
        # general explanation were asked for — and acts on that accordingly
        return handle_explain_request(user_message, text)
    elif any(word in msg_lower for word in ["quiz", "اختبار", "كويز"]):
        return generate_quiz(text)
    else:
        return rag_query(coll, user_message)


def render_map():
    text = APP_STATE.get("extracted_text", "")
    if not text:
        return "<h3 style='color:#f1f5f9;'>⚠️ Please upload a file first.</h3>"
    html_file = generate_mindmap_html(text)
    with open(html_file, "r") as f:
        return f.read()


CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700&display=swap');

* { font-family: 'Tajawal', sans-serif !important; }

.gradio-container {
    background: linear-gradient(160deg, #0f172a 0%, #1e293b 100%) !important;
}

#header-banner {
    background: linear-gradient(90deg, #6366f1 0%, #8b5cf6 60%, #ec4899 100%);
    padding: 28px 32px;
    border-radius: 18px;
    margin-bottom: 18px;
    box-shadow: 0 10px 30px rgba(99, 102, 241, 0.35);
}
#header-banner h1 {
    color: white !important;
    margin: 0 0 6px 0 !important;
}
#header-banner p {
    color: rgba(255,255,255,0.9) !important;
    margin: 0 !important;
}

.upload-card, .chat-card {
    border-radius: 16px !important;
    border: 1px solid rgba(148,163,184,0.15) !important;
}

#process-btn {
    background: linear-gradient(90deg, #6366f1, #8b5cf6) !important;
    border: none !important;
    font-weight: 700 !important;
}

#map-btn {
    background: linear-gradient(90deg, #10b981, #059669) !important;
    border: none !important;
    font-weight: 700 !important;
}
"""

with gr.Blocks(title="StudyMate - Local AI", theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="violet"), css=CUSTOM_CSS) as demo:
    gr.HTML("""
    <div id="header-banner">
        <h1>🎓 StudyMate</h1>
        <p>Your smart study assistant — runs entirely locally, no external API required 🚀</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, elem_classes="upload-card"):
            gr.Markdown("### 📤 Upload Your Lecture")
            file_input = gr.File(label="PDF or image (PNG/JPG)")
            upload_btn = gr.Button("⚙️ Process File", elem_id="process-btn")
            status_box = gr.Textbox(label="Status", interactive=False)
            preview_box = gr.Textbox(label="👀 Extracted Text Preview", interactive=False, lines=6)

        with gr.Column(scale=2, elem_classes="chat-card"):
            with gr.Tabs():
                with gr.TabItem("💬 Chat with the Assistant (LLM)"):
                    gr.Markdown("Type **explain / summarize** to get a summary, **quiz** to generate a quiz, or ask any specific question about the lecture. Works in both English and Arabic.")
                    chatbot = gr.ChatInterface(fn=user_chat_router)

                with gr.TabItem("🗺️ Concept Mind Map (LLM)"):
                    generate_map_btn = gr.Button("🧠 Generate Mind Map", elem_id="map-btn")
                    map_html = gr.HTML()

                with gr.TabItem("🎓 Classical NLP Techniques (No LLM)"):
                    gr.Markdown(
                        "This section is completely independent of Qwen/BART — extractive summarization with "
                        "**TextRank** and **Cloze (fill-in-the-blank)** quiz generation via POS/NER, to "
                        "demonstrate classical NLP techniques alongside the LLM-based part."
                    )
                    with gr.Row():
                        textrank_btn = gr.Button("📝 TextRank Summary", elem_id="process-btn")
                        cloze_btn = gr.Button("❓ Cloze Quiz", elem_id="map-btn")
                    classic_output = gr.Textbox(label="Result", lines=10, interactive=False)

    def run_textrank():
        text = APP_STATE.get("extracted_text", "")
        if not text:
            return "⚠️ Please upload a file first."
        return textrank_summarize(text, num_sentences=5)

    def run_cloze():
        text = APP_STATE.get("extracted_text", "")
        if not text:
            return "⚠️ Please upload a file first."
        return generate_cloze_quiz(text, num_questions=3)

    # Callbacks
    upload_btn.click(fn=handle_file_upload, inputs=[file_input], outputs=[status_box, preview_box])
    generate_map_btn.click(fn=render_map, inputs=[], outputs=[map_html])
    textrank_btn.click(fn=run_textrank, inputs=[], outputs=[classic_output])
    cloze_btn.click(fn=run_cloze, inputs=[], outputs=[classic_output])

demo.launch(share=True)
